# Exercise 3: prompt experiment – Google Flow and OpenAI API

## Tasks

1. **One prompt:** write a prompt and create a picture in Flow.
2. **Vary:** change one building block of the prompt and create again.
3. **Repeat:** create several pictures for each version.
4. **Compare:** rate one feature of the pictures – by eye or with the OpenAI API.

Images in Flow are free. Do not create videos yet.

## Setup

Select the **Python 3.11** kernel. The pictures are created in **Google Flow** (a web app, no code). The OpenAI API key is only needed for the optional part at the end.

In [ ]:
# Standard Python libraries.
import os
import sys
from pathlib import Path

# Libraries installed from requirements.txt.
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Work inside this exercise folder, wherever Jupyter starts.
if not Path("data").exists():  # we are inside an exercise folder
    os.chdir("..")  # go to the main folder
os.chdir("03_Prompt_Experiment")
sys.path.append("..")  # helper.py is in the main folder

## 1. Write a prompt from building blocks

Describe what should be visible. Copy the printed prompt into Flow and create a picture.

In [ ]:
# The building blocks of the prompt.
blocks_a = {
    "subject": "a red wooden rowboat",
    "setting": "on a mountain lake",
    "light": "at dawn, soft light",
    "composition": "wide shot",
}

# Fill the template with the blocks.
TEMPLATE = "{subject}, {setting}, {light}, {composition}"
print("A:", TEMPLATE.format(**blocks_a))

## 2. Change one building block and create again

Only **one** block changes, the rest stays the same – so you know what caused the difference.

In [ ]:
# Copy the blocks and change only the composition.
blocks_b = blocks_a.copy()
blocks_b["composition"] = "wide shot, boat in the left third"
print("B:", TEMPLATE.format(**blocks_b))

# The feature you will rate on every picture.
FEATURE = "The center of the red rowboat is in the left third of the image."

## 3. Create three pictures per version and upload them

Use the same Flow image model and aspect ratio for all pictures. Upload them into the folder `flow_images/` next to this notebook and name them:

`variant_a_1`, `variant_a_2`, `variant_a_3`, `variant_b_1`, `variant_b_2`, `variant_b_3` (`.jpg` or `.png`)

In [ ]:
# Create the upload folder.
folder = Path("flow_images")
folder.mkdir(exist_ok=True)

# Find the six expected files and stop if one is missing or doubled.
names = [
    "variant_a_1", "variant_a_2", "variant_a_3",
    "variant_b_1", "variant_b_2", "variant_b_3",
]
paths = []
for name in names:
    found = list(folder.glob(name + ".jpg")) + list(folder.glob(name + ".png"))
    message = f"flow_images/ needs exactly one {name}.jpg or {name}.png"
    assert len(found) == 1, message
    paths.append(found[0])
print("All 6 images found.")

## 4. Compare the versions by eye

In [ ]:
# Show version A in the top row and version B in the bottom row.
fig, axes = plt.subplots(2, 3, figsize=(9, 4))
for i in range(6):
    ax = axes.flat[i]
    ax.imshow(Image.open(paths[i]))
    ax.set_title(names[i])
    ax.axis("off")
plt.show()

In [ ]:
# One row per picture.
ratings = pd.DataFrame({
    "name": names,
    "version": ["A", "A", "A", "B", "B", "B"],
})

# Your rating per picture, in the order shown: True if the feature is visible.
# Replace these example values with your own.
ratings["me"] = [False, True, False, True, True, False]

# True counts as 1, False as 0 – so the mean is the share of "True".
ratings.groupby("version")["me"].mean()

## Optional: let the OpenAI API rate the same feature

The same yes/no question for all six pictures, sent to the **OpenAI API** (model `gpt-6-luna`). Does it agree with you?

In [ ]:
# Libraries for the OpenAI API part.
from pydantic import BaseModel

# Shared helpers from helper.py (this connects to the OpenAI API).
from helper import ask

In [ ]:
# The answer format: yes or no, plus a short reason.
class Rating(BaseModel):
    matches: bool
    reason: str


# The same question for every picture.
PROMPT = (
    f"Is this true? {FEATURE} "
    "If there is no boat, answer false. Give a short reason."
)

In [ ]:
# Ask the OpenAI model about every picture.
api = []
for path in paths:
    api.append(ask(PROMPT, Rating, [path])["matches"])

# Your rating and the OpenAI rating side by side.
ratings["api"] = api
display(ratings)
ratings.groupby("version")[["me", "api"]].mean()

**Check yourself:** Why does the same prompt give different pictures – and what does that mean for your experiment?

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')